# Search Evaluation (Module 4, Lesson 04)

For each ground-truth question we run search and check whether the correct
document ID appears in the top-k results. The output is a **relevance list**
per question — a row of `0`/`1` values, one per retrieved document.

Lesson 05 turns these lists into Hit Rate and MRR.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "eval" / "search_eval.py").exists():
    root = root.parent
sys.path[:0] = [str(root), str(root / "src")]

from eval.search_eval import (
    compute_relevance,
    compute_relevance_total,
    load_course_documents,
    load_ground_truth,
    make_keyword_search,
    make_vector_search,
)

ground_truth = load_ground_truth()
documents = load_course_documents()

print(f"ground truth questions: {len(ground_truth)}")
print(f"course documents: {len(documents)}")

## 1. Set up search

The course wraps minsearch in `text_search(query)`. Here we use the same
pattern with `KeywordRetriever` — a function that takes a query string and
returns ranked documents. Later we can swap in `vector_search` without
changing the relevance logic.

In [ ]:
K = 5

text_search = make_keyword_search(documents, k=K)
vector_search = make_vector_search(k=K)

## 2. One question — build a relevance list

Ground-truth record: `question` + `document` (the correct doc ID).

Run search, then mark each result `1` if its `id` matches the correct
document, else `0`.

In [ ]:
q = ground_truth[0]
doc_id = q["document"]
results = text_search(q["question"])

print(q["question"])
print()

for doc in results:
    match = doc["id"] == doc_id
    print(f'{doc["id"]} == {doc_id}: {match}')

relevance = compute_relevance(q, text_search)
print()
print("relevance list:", relevance)

## 3. Generic relevance functions

`compute_relevance` works with any search function — keyword, vector, or
hybrid. Only the search backend changes; the evaluation logic stays the same.

In [ ]:
for idx in [0, 11, 50]:
    q = ground_truth[idx]
    print(q["question"])
    print(compute_relevance(q, text_search))
    print()

## 4. Sample then full dataset

Start with 15 questions to verify the pipeline, then run all 395.

In [ ]:
relevance_total_sample = compute_relevance_total(
    ground_truth,
    text_search,
    sample=15,
)
relevance_total_sample

In [ ]:
relevance_total_keyword = compute_relevance_total(ground_truth, text_search)
len(relevance_total_keyword)

## 5. Same evaluation for vector search

`vector_search` needs Postgres + PGVector indexed (`make ingest`). The
relevance functions are unchanged — swap the search function only.

In [ ]:
relevance_total_vector = compute_relevance_total(ground_truth, vector_search)
len(relevance_total_vector)